## Литература
Application of machine learning algorithms for population forecasting, 2021

The target variable is determined as the total population value of 2017. Валидацию делали на Турции.

Здесь мы будем предсказывать на 10 лет вперёд, целевая переменная будет численность населения 2013-2022 включительно. Здесь мы делаем валидацию на России без учёта Крыма.

## Baseline
### Naïve (Last Observation)

Официальный базовый метод временных рядов (Hyndman & Athanasopoulos, *Forecasting*).

**Формула:**

\[
\hat{y}_{t+h} = y_t
\]

Прогноз равен последнему известному значению.

**Почему это baseline:**  
Любая модель должна быть лучше сценария «ничего не меняется». В демографии тренды часто инерционные, поэтому такой тест особенно важен.

Синонимы: **Naïve Forecast**, **Last Value Forecast**.

---

### 2) Linear Trend (OLS Extrapolation)

Классический демографический базовый метод для сравнения со сложными моделями.

**Модель:**

\[
y_t = a + b t
\]

Подгоняется по обучающему отрезку, затем линейно экстраполируется на годы \(t+1,\ldots,t+H\).

**Почему используется:**  
Демографические организации (UN, Eurostat) приводят линейную аппроксимацию как технический среднесрочный ориентир и минимальный структурный baseline.

Название: **Linear Trend Extrapolation**.


### Метрики для сравнения моделей
Взяты из статьи 
#### 1) MAPE (Mean Absolute Percentage Error)

\[
\text{MAPE} = \frac{1}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right|
\]


#### 2) RMSE (Root Mean Squared Error)
\[
\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}
\]


### 3) SDEV ошибок (стандартное отклонение абсолютных ошибок)
Нужно для анализа стабильности модели между странами и годами.  
MAPE может быть одинаковым у двух моделей, но одна стабильна, а другая «стреляющая» — SDEV это показывает.
\[
\text{SDEV} = \sqrt{\frac{1}{n}\sum_{i=1}^n (e_i - \bar{e})^2}
\]
где \(e_i = |y_i - \hat{y}_i|\).


### 3 Rolling Windows

Для оценки стабильности моделей по времени используются три окна:

1. **Окно A:** train до 1999, тест 2000–2009  
2. **Окно B:** train до 2004, тест 2005–2014  
3. **Окно C:** train до 2012, тест 2013–2022  

Каждое окно покрывает 10 лет прогнозирования. Три окна позволяют проверить, насколько модели устойчивы к различным историческим периодам, выявить зависимость качества прогнозов от трендов и колебаний населения. Такой подход имитирует скользящий forecast, что часто применяют в демографии для оценки краткосрочной и среднесрочной точности моделей.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("output/basic_with_cohort.csv")
#df = pd.read_csv("output/merged_worldbank_long.csv")
def naive_forecast(train_series, horizon):
    last_value = train_series.iloc[-1]
    return np.array([last_value] * horizon)

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

def rmse_perc(y_true, y_pred):
    rmse_val = np.sqrt(np.mean((y_true - y_pred)**2))
    return rmse_val / np.mean(y_true)

def sdev_perc(y_true, y_pred):
    e = np.abs(y_true - y_pred)
    return e.std() / np.mean(y_true)

models_results = {'naive': {}}

# Определяем 3 окна
windows = [
    ('A', 1999, 10),  # train до 1999, тест 2000–2009
    ('B', 2004, 10),  # train до 2004, тест 2005–2014
    ('C', 2012, 10)   # train до 2012, тест 2013–2022
]

for window_name, train_end_year, horizon in windows:
    all_mape, all_rmse, all_sdev = [], [], []
    models_results['naive'][window_name] = {}
    
    for country in df['country_code'].unique():
        df_c = df[df['country_code'] == country].sort_values('year')
        train = df_c[df_c['year'] <= train_end_year]
        test = df_c[(df_c['year'] > train_end_year) & (df_c['year'] <= train_end_year + horizon)]
        y_train = train['population']
        y_test = test['population'].values
        y_pred = naive_forecast(y_train, len(y_test))
        
        r = {
            'MAPE': mape(y_test, y_pred),
            'RMSE': rmse_perc(y_test, y_pred),
            'SDEV': sdev_perc(y_test, y_pred),
            'pred': y_pred,
            'true': y_test
        }
        models_results['naive'][window_name][country] = r
        
        all_mape.append(r['MAPE'])
        all_rmse.append(r['RMSE'])
        all_sdev.append(r['SDEV'])
    
    print(f"Naïve baseline - window {window_name} (all countries):")
    print(f"MAPE: {np.mean(all_mape):.3f}, RMSE: {np.mean(all_rmse):.3f}, SDEV: {np.mean(all_sdev):.3f}\n")


Naïve baseline - window A (all countries):
MAPE: 0.085, RMSE: 0.101, SDEV: 0.050

Naïve baseline - window B (all countries):
MAPE: 0.086, RMSE: 0.102, SDEV: 0.049

Naïve baseline - window C (all countries):
MAPE: 0.081, RMSE: 0.094, SDEV: 0.043



In [19]:
def print_model_summary(model_name):
    for window_name in models_results[model_name]:
        all_mape = [r['MAPE'] for r in models_results[model_name][window_name].values()]
        all_rmse = [r['RMSE'] for r in models_results[model_name][window_name].values()]
        all_sdev = [r['SDEV'] for r in models_results[model_name][window_name].values()]
        print(f"model {model_name.title()} - window {window_name}:")
        print(f"  MAPE: {np.mean(all_mape):.3f}, RMSE: {np.mean(all_rmse):.3f}, SDEV: {np.mean(all_sdev):.3f}\n")
print_model_summary('naive')

model Naive - window A:
  MAPE: 0.085, RMSE: 0.101, SDEV: 0.050

model Naive - window B:
  MAPE: 0.086, RMSE: 0.102, SDEV: 0.049

model Naive - window C:
  MAPE: 0.081, RMSE: 0.094, SDEV: 0.043



In [20]:
from sklearn.linear_model import LinearRegression

models_results['linear_trend'] = {}

for window_name, train_end_year, horizon in windows:
    models_results['linear_trend'][window_name] = {}
    
    for country in df['country_code'].unique():
        df_c = df[df['country_code'] == country].sort_values('year')
        train = df_c[df_c['year'] <= train_end_year]
        test = df_c[(df_c['year'] > train_end_year) & (df_c['year'] <= train_end_year + horizon)]
        
        X_train = train['year'].values.reshape(-1, 1)
        y_train = train['population'].values
        X_test = test['year'].values.reshape(-1, 1)
        y_test = test['population'].values
        
        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        models_results['linear_trend'][window_name][country] = {
            'MAPE': mape(y_test, y_pred),
            'RMSE': rmse_perc(y_test, y_pred),
            'SDEV': sdev_perc(y_test, y_pred),
            'pred': y_pred,
            'true': y_test
        }

print_model_summary('linear_trend')


model Linear_Trend - window A:
  MAPE: 0.074, RMSE: 0.080, SDEV: 0.026

model Linear_Trend - window B:
  MAPE: 0.085, RMSE: 0.091, SDEV: 0.028

model Linear_Trend - window C:
  MAPE: 0.099, RMSE: 0.103, SDEV: 0.025



In [21]:
from xgboost import XGBRegressor

features = ['age_0_14_share', 'age_65_plus_share', 'crude_birth_rate', 'crude_death_rate',
            'fertility_rate', 'life_expectancy', 'net_migration', 'urban_share', 'gdppc']

models_results['xgboost'] = {}

for window_name, train_end_year, horizon in windows:
    models_results['xgboost'][window_name] = {}
    
    for country in df['country_code'].unique():
        df_c = df[df['country_code'] == country].sort_values('year')
        train = df_c[df_c['year'] <= train_end_year]
        test = df_c[(df_c['year'] > train_end_year) & (df_c['year'] <= train_end_year + horizon)]
        
        X_train = train[features]
        y_train = train['population'].values
        X_test = test[features]
        y_test = test['population'].values
        
        model = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        models_results['xgboost'][window_name][country] = {
            'MAPE': mape(y_test, y_pred),
            'RMSE': rmse_perc(y_test, y_pred),
            'SDEV': sdev_perc(y_test, y_pred),
            'pred': y_pred,
            'true': y_test
        }

print_model_summary('xgboost')


model Xgboost - window A:
  MAPE: 0.109, RMSE: 0.127, SDEV: 0.057

model Xgboost - window B:
  MAPE: 0.103, RMSE: 0.121, SDEV: 0.057

model Xgboost - window C:
  MAPE: 0.105, RMSE: 0.120, SDEV: 0.053



In [ ]:
from prophet import Prophet

models_results['prophet'] = {}

for window_name, train_end_year, horizon in windows:
    models_results['prophet'][window_name] = {}
    
    for country in df['country_code'].unique():
        df_c = df[df['country_code'] == country].sort_values('year')
        train = df_c[df_c['year'] <= train_end_year]
        test = df_c[(df_c['year'] > train_end_year) & (df_c['year'] <= train_end_year + horizon)]
        
        # Prophet требует колонки 'ds' (дата) и 'y' (значение)
        train_prophet = train[['year', 'population']].rename(columns={'year': 'ds', 'population': 'y'})
        train_prophet['ds'] = pd.to_datetime(train_prophet['ds'], format='%Y')
        
        model = Prophet(yearly_seasonality=False, daily_seasonality=False, weekly_seasonality=False)
        model.fit(train_prophet)
        
        future = pd.DataFrame({'ds': pd.date_range(start=str(train_end_year+1), periods=horizon, freq='YE')})
        forecast = model.predict(future)
        y_pred = forecast['yhat'].values
        y_test = test['population'].values
        
        models_results['prophet'][window_name][country] = {
            'MAPE': mape(y_test, y_pred),
            'RMSE': rmse_perc(y_test, y_pred),
            'SDEV': sdev_perc(y_test, y_pred),
            'pred': y_pred,
            'true': y_test
        }

print_model_summary('prophet')


23:16:49 - cmdstanpy - INFO - Chain [1] start processing
23:16:49 - cmdstanpy - INFO - Chain [1] done processing
23:16:50 - cmdstanpy - INFO - Chain [1] start processing
23:16:50 - cmdstanpy - INFO - Chain [1] done processing
23:16:50 - cmdstanpy - INFO - Chain [1] start processing
23:16:50 - cmdstanpy - INFO - Chain [1] done processing
23:16:50 - cmdstanpy - INFO - Chain [1] start processing
23:16:51 - cmdstanpy - INFO - Chain [1] done processing
23:16:51 - cmdstanpy - INFO - Chain [1] start processing
23:16:51 - cmdstanpy - INFO - Chain [1] done processing
23:16:51 - cmdstanpy - INFO - Chain [1] start processing
23:16:51 - cmdstanpy - INFO - Chain [1] done processing
23:16:51 - cmdstanpy - INFO - Chain [1] start processing
23:16:52 - cmdstanpy - INFO - Chain [1] done processing
23:16:52 - cmdstanpy - INFO - Chain [1] start processing
23:16:52 - cmdstanpy - INFO - Chain [1] done processing
23:16:52 - cmdstanpy - INFO - Chain [1] start processing
23:16:52 - cmdstanpy - INFO - Chain [1]

In [ ]:
models_results['cohort_component'] = {}

for window_name, train_end_year, horizon in windows:
    models_results['cohort_component'][window_name] = {}
    
    for country in df['country_code'].unique():
        df_c = df[df['country_code'] == country].sort_values('year')
        train = df_c[df_c['year'] <= train_end_year]
        test = df_c[(df_c['year'] > train_end_year) & (df_c['year'] <= train_end_year + horizon)]
        
        # численность когорт на последний год train
        pop_total = train['population'].iloc[-1]
        pop_0_14 = pop_total * train['age_0_14_share'].iloc[-1]
        pop_15_64 = pop_total - pop_0_14 - pop_total * train['age_65_plus_share'].iloc[-1]
        pop_65_plus = pop_total * train['age_65_plus_share'].iloc[-1]
        
        y_pred = []
        for i in range(horizon):
            # простая имитация CCM: выживаемость + рождаемость + миграция
            births = pop_15_64 * train['fertility_rate'].iloc[-1] / 1000  # упрощение
            deaths = pop_total * train['crude_death_rate'].iloc[-1] / 1000
            net_mig = train['net_migration'].iloc[-1]
            
            pop_total = pop_total + births - deaths + net_mig
            y_pred.append(pop_total)
        
        y_test = test['population'].values
        
        models_results['cohort_component'][window_name][country] = {
            'MAPE': mape(y_test, y_pred),
            'RMSE': rmse_perc(y_test, y_pred),
            'SDEV': sdev_perc(y_test, y_pred),
            'pred': y_pred,
            'true': y_test
        }

print_model_summary('cohort_component')


model Cohort_Component - window A:
  MAPE: 0.801, RMSE: 0.927, SDEV: 0.425

model Cohort_Component - window B:
  MAPE: 0.732, RMSE: 0.847, SDEV: 0.388

model Cohort_Component - window C:
  MAPE: 0.663, RMSE: 0.763, SDEV: 0.348

